# Hybrid ML–DL Crime Severity Classification — Final Version (Row-Level + District-Grouped Split)

**Why this version exists:**
- The aggregated (district-year) version had too few rows (~3,000) for the deep hybrids, especially the Bi-LSTM ones.
- The row-level (no aggregation) version had enough rows (~30,000) but leaked information: 10 rows share the same district-year (one per crime type), sharing `Population`, `State`, similar `Chargesheet_Rate`. A random row split let "sibling" rows split across train/test.

**The fix:** keep the row-level data (full sample size), but split by **District** instead of by row. Every row for a given district — across all years and all crime types — goes entirely into train or entirely into test. This removes the leakage, and it also means all four hybrids (A, B, C, D) are evaluated on the *same* held-out districts, so the final comparison is genuinely apples-to-apples (in the original notebook, the tabular hybrids and the sequence hybrids were scored on different splits).

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('TensorFlow:', tf.__version__)
print('XGBoost / LightGBM loaded OK')


## 2. Load Data (raw row grain, no aggregation)

In [ ]:
df = pd.read_csv('india_district_crime_2014_2023_30k.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
print(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nStates:', df.State.nunique(), '| Districts:', df.District.nunique(),
      '| Years:', df.Year.min(),'-', df.Year.max(), '| Crime types:', df.Crime_Type.nunique())
df.describe().T


## 3. Feature Engineering (row-level)

Same target formula as before, computed per row (District, Year, Crime_Type) instead of per district-year:

| Component | Computed on | Weight |
|---|---|---|
| Severity-weighted crime rate | `Weighted_Cases / Population * 100000`, per row | 0.45 |
| Raw crime rate | `Crime_Rate_per_100k`, already per row | 0.20 |
| YoY growth in weighted crime rate | per (District, Crime_Type) trajectory | 0.15 |
| Impunity (1 − conviction rate) | `Convictions / Cases_Reported`, per row | 0.20 |

> **Target-leakage guard:** `Weighted_Crime_Rate`, `Crime_Rate_per_100k`, `YoY_Growth`, `Conviction_Rate` build the target and are excluded from the model-facing features.

In [ ]:
severity_weights = {
    'Murder': 10, 'Dowry Deaths': 9, 'Rape': 9, 'Kidnapping': 7,
    'Robbery': 6, 'Assault': 5, 'Burglary': 4, 'Theft': 3,
    'Fraud': 3, 'Cybercrime': 2
}
df['Severity_Weight'] = df['Crime_Type'].map(severity_weights)
df['Weighted_Cases'] = df['Cases_Reported'] * df['Severity_Weight']
df['Weighted_Crime_Rate'] = df['Weighted_Cases'] / df['Population'] * 100000
df['Conviction_Rate'] = df['Convictions'] / df['Cases_Reported']
df['Chargesheet_Rate'] = df['Chargesheeted'] / df['Cases_Reported']

df = df.sort_values(['District', 'Crime_Type', 'Year']).reset_index(drop=True)
print('Row-level shape (no aggregation):', df.shape)
df.head()


In [ ]:
# ---- TARGET-ONLY temporal feature, per (District, Crime_Type) trajectory across years ----
def target_temporal_feats(g):
    g = g.sort_values('Year').copy()
    g['YoY_Growth'] = g['Weighted_Crime_Rate'].pct_change().fillna(0)
    return g

df = df.groupby(['District', 'Crime_Type']).apply(target_temporal_feats, include_groups=False)
df = df.reset_index(level=[0, 1]).reset_index(drop=True)

# ---- MODEL-FACING temporal features, computed on Cases_Reported (independent of target formula) ----
def model_temporal_feats(g):
    g = g.sort_values('Year').copy()
    g['Cases_YoY'] = g['Cases_Reported'].pct_change().fillna(0)
    g['Cases_Roll3_Mean'] = g['Cases_Reported'].rolling(3, min_periods=1).mean()
    g['Cases_Roll3_Std'] = g['Cases_Reported'].rolling(3, min_periods=1).std().fillna(0)
    yrs = g['Year'].values
    vals = g['Cases_Reported'].values
    slopes = [0.0 if i < 1 else np.polyfit(yrs[:i+1], vals[:i+1], 1)[0] for i in range(len(g))]
    g['Cases_Trend_Slope'] = slopes
    return g

df = df.groupby(['District', 'Crime_Type']).apply(model_temporal_feats, include_groups=False)
df = df.reset_index(level=[0, 1]).reset_index(drop=True)

# ---- State-relative (cross-sectional) features, computed on Cases_Reported (NOT a target ingredient) ----
df['State_Year_Z'] = df.groupby(['State', 'Year'])['Cases_Reported'] \
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
df['State_Year_Rank_Pct'] = df.groupby(['State', 'Year'])['Cases_Reported'].rank(pct=True)


In [ ]:
minmax = MinMaxScaler()
norm = pd.DataFrame(
    minmax.fit_transform(df[['Weighted_Crime_Rate', 'Crime_Rate_per_100k', 'YoY_Growth']]),
    columns=['n_wcr', 'n_acr', 'n_yoy']
)
impunity = 1 - df['Conviction_Rate'].fillna(0)
norm['n_imp'] = MinMaxScaler().fit_transform(impunity.values.reshape(-1, 1))

df['Severity_Score'] = (
    0.45 * norm['n_wcr'] +
    0.20 * norm['n_acr'] +
    0.15 * norm['n_yoy'].clip(0, 1) +
    0.20 * norm['n_imp']
)

df['Severity_Class'] = pd.qcut(df['Severity_Score'], 4, labels=['Low', 'Moderate', 'High', 'Severe'])

print(df['Severity_Class'].value_counts())
plt.figure(figsize=(6,4))
df['Severity_Class'].value_counts().reindex(['Low','Moderate','High','Severe']).plot(kind='bar', color='teal')
plt.title('Target Class Distribution (Row-Level Severity)')
plt.ylabel('Count')
plt.show()


## 4. Preprocessing — District-Grouped Split (the key fix)

`StratifiedGroupKFold` splits by `District` (the group) while still trying to keep class balance close across the split. This guarantees no district's rows appear in both train and test, and it's the split every hybrid below will reuse.

In [ ]:
model_df = df.copy()

feature_cols = ['Cases_Reported', 'Chargesheet_Rate', 'Cases_YoY', 'Cases_Roll3_Mean',
                'Cases_Roll3_Std', 'Cases_Trend_Slope', 'State_Year_Z', 'State_Year_Rank_Pct',
                'Chargesheeted', 'Convictions', 'Population']

state_dummies = pd.get_dummies(model_df['State'], prefix='State')
crime_dummies = pd.get_dummies(model_df['Crime_Type'], prefix='Crime')
X = pd.concat([model_df[feature_cols], state_dummies, crime_dummies], axis=1)

le = LabelEncoder()
y = le.fit_transform(model_df['Severity_Class'])
class_names = le.classes_
print('Classes:', class_names)

# ---- District-grouped split: no district's rows can appear in both train and test ----
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(X, y, groups=model_df['District'].values))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_districts = set(model_df.iloc[train_idx]['District'])
test_districts = set(model_df.iloc[test_idx]['District'])
assert train_districts.isdisjoint(test_districts), 'Leakage: a district appears in both splits!'
print(f'Train districts: {len(train_districts)} | Test districts: {len(test_districts)}')
print('Train rows:', X_train.shape, ' Test rows:', X_test.shape)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

n_classes = len(class_names)


In [ ]:
# ---- Central evaluation utility used for every hybrid model ----
results = {}

def evaluate_model(name, y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_m = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    if y_proba is not None:
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    results[name] = {
        'Accuracy': acc, 'Precision_macro': prec_m, 'Recall_macro': rec_m,
        'F1_macro': f1_m, 'F1_weighted': f1_w, 'ROC_AUC_ovr': auc
    }
    print(f'--- {name} ---')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    return results[name]

def plot_confusion(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix — {name}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.show()


## 5. Hybrid ML–DL Architectures

All four hybrids below use the **same** `train_districts` / `test_districts` split, so the Section 6 comparison is a fair, apples-to-apples ranking.

### Hybrid A — Stacked Ensemble (XGBoost + LightGBM + Random Forest → Neural Meta-Learner)

In [ ]:
base_models = {
    'xgb': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, objective='multi:softprob', num_class=n_classes,
                          eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1),
    'lgbm': LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                            colsample_bytree=0.8, random_state=RANDOM_STATE, verbosity=-1),
    'rf': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_probs, test_probs = {}, {}
for name, m in base_models.items():
    oof_probs[name] = cross_val_predict(m, X_train, y_train, cv=skf, method='predict_proba', n_jobs=-1)
    m.fit(X_train, y_train)
    test_probs[name] = m.predict_proba(X_test)

meta_X_train = np.hstack([oof_probs[k] for k in base_models])
meta_X_test = np.hstack([test_probs[k] for k in base_models])
print('Meta-feature shape:', meta_X_train.shape)


In [ ]:
def build_meta_learner(input_dim, n_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

meta_nn = build_meta_learner(meta_X_train.shape[1], n_classes)
es = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss')
meta_nn.fit(meta_X_train, y_train, validation_split=0.15, epochs=100, batch_size=32, callbacks=[es], verbose=0)

proba = meta_nn.predict(meta_X_test, verbose=0)
pred = np.argmax(proba, axis=1)
evaluate_model('Hybrid A: Stacked (XGB+LGBM+RF -> NN)', y_test, pred, proba)
plot_confusion('Hybrid A: Stacked Ensemble', y_test, pred)


### Building sequences for Hybrid B and Hybrid D

Sequences are built per (District, Crime_Type) pair (10 years each). Each sequence is assigned to train or test based on whether its District is in `train_districts` or `test_districts` — the **same split as the tabular models above**, not a separate random split.

In [ ]:
seq_feature_cols = feature_cols

X_seq_list, y_seq_list, seq_district = [], [], []
for (district, crime_type), sub in model_df.groupby(['District', 'Crime_Type']):
    sub = sub.sort_values('Year')
    if len(sub) != 10:          # keep only complete 10-year trajectories
        continue
    X_seq_list.append(sub[seq_feature_cols].values)
    y_seq_list.append(le.transform(sub['Severity_Class']))
    seq_district.append(district)

X_seq = np.array(X_seq_list)          # (n_groups, 10, n_features)
y_seq = np.array(y_seq_list)          # (n_groups, 10)
seq_district = np.array(seq_district)
print('Sequence tensor shape (District x Crime_Type groups):', X_seq.shape)

train_mask = np.isin(seq_district, list(train_districts))
test_mask = np.isin(seq_district, list(test_districts))
print('Sequence groups -> train:', train_mask.sum(), ' test:', test_mask.sum(),
      ' dropped (incomplete trajectory):', len(seq_district) - train_mask.sum() - test_mask.sum())

X_seq_train_raw, X_seq_test_raw = X_seq[train_mask], X_seq[test_mask]
y_seq_train, y_seq_test = y_seq[train_mask], y_seq[test_mask]

seq_scaler = StandardScaler()
seq_scaler.fit(X_seq_train_raw.reshape(-1, X_seq.shape[-1]))
X_seq_train = seq_scaler.transform(X_seq_train_raw.reshape(-1, X_seq.shape[-1])).reshape(X_seq_train_raw.shape)
X_seq_test = seq_scaler.transform(X_seq_test_raw.reshape(-1, X_seq.shape[-1])).reshape(X_seq_test_raw.shape)


### Hybrid B — CNN + Bi-LSTM

In [ ]:
def build_cnn_bilstm(timesteps, n_features, n_classes):
    inp = layers.Input(shape=(timesteps, n_features))
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.TimeDistributed(layers.Dense(32, activation='relu'))(x)
    out = layers.TimeDistributed(layers.Dense(n_classes, activation='softmax'))(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_bilstm = build_cnn_bilstm(X_seq.shape[1], X_seq.shape[2], n_classes)
es = keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss')
hist = cnn_bilstm.fit(X_seq_train, y_seq_train, validation_split=0.15, epochs=150, batch_size=16,
                       callbacks=[es], verbose=0)

proba_seq = cnn_bilstm.predict(X_seq_test, verbose=0)
pred_seq = np.argmax(proba_seq, axis=-1)
y_true_flat = y_seq_test.flatten()
y_pred_flat = pred_seq.flatten()
y_proba_flat = proba_seq.reshape(-1, n_classes)

evaluate_model('Hybrid B: CNN-BiLSTM', y_true_flat, y_pred_flat, y_proba_flat)
plot_confusion('Hybrid B: CNN-BiLSTM', y_true_flat, y_pred_flat)


### Hybrid C — Deep Autoencoder + XGBoost

In [ ]:
def build_autoencoder(input_dim, latent_dim=8):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='relu')(inp)
    x = layers.Dense(16, activation='relu')(x)
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)
    x = layers.Dense(16, activation='relu')(latent)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(input_dim, activation='linear')(x)
    ae = keras.Model(inp, out)
    encoder = keras.Model(inp, latent)
    ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return ae, encoder

ae, encoder = build_autoencoder(X_train_s.shape[1], latent_dim=8)
es = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss')
ae.fit(X_train_s.values, X_train_s.values, validation_split=0.15, epochs=150, batch_size=32,
       callbacks=[es], verbose=0)

Z_train = encoder.predict(X_train_s.values, verbose=0)
Z_test = encoder.predict(X_test_s.values, verbose=0)

Z_train_full = np.hstack([Z_train, X_train.values])
Z_test_full = np.hstack([Z_test, X_test.values])

xgb_ae = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, objective='multi:softprob', num_class=n_classes,
                        eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1)
xgb_ae.fit(Z_train_full, y_train)
pred = xgb_ae.predict(Z_test_full); proba = xgb_ae.predict_proba(Z_test_full)
evaluate_model('Hybrid C: Autoencoder + XGBoost', y_test, pred, proba)
plot_confusion('Hybrid C: Autoencoder + XGBoost', y_test, pred)


### Hybrid D — Bi-LSTM Embedding + XGBoost

In [ ]:
def build_bilstm_encoder(timesteps, n_features, n_classes):
    inp = layers.Input(shape=(timesteps, n_features))
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x)
    emb = layers.TimeDistributed(layers.Dense(32, activation='relu'), name='embedding')(x)
    out = layers.TimeDistributed(layers.Dense(n_classes, activation='softmax'))(emb)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

bilstm_for_embed = build_bilstm_encoder(X_seq.shape[1], X_seq.shape[2], n_classes)
es = keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss')
bilstm_for_embed.fit(X_seq_train, y_seq_train, validation_split=0.15, epochs=150, batch_size=16,
                      callbacks=[es], verbose=0)

embed_model = keras.Model(inputs=bilstm_for_embed.input,
                           outputs=bilstm_for_embed.get_layer('embedding').output)

emb_train_seq = embed_model.predict(X_seq_train, verbose=0)
emb_test_seq = embed_model.predict(X_seq_test, verbose=0)

emb_train_flat = emb_train_seq.reshape(-1, emb_train_seq.shape[-1])
emb_test_flat = emb_test_seq.reshape(-1, emb_test_seq.shape[-1])
y_train_seq_flat = y_seq_train.flatten()
y_test_seq_flat = y_seq_test.flatten()

xgb_lstm = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, objective='multi:softprob', num_class=n_classes,
                          eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1)
xgb_lstm.fit(emb_train_flat, y_train_seq_flat)
pred = xgb_lstm.predict(emb_test_flat); proba = xgb_lstm.predict_proba(emb_test_flat)
evaluate_model('Hybrid D: BiLSTM-Embedding + XGBoost', y_test_seq_flat, pred, proba)
plot_confusion('Hybrid D: BiLSTM-Embedding + XGBoost', y_test_seq_flat, pred)


## 6. Hybrid Model Comparison — which one is genuinely best

All four rows in this table are evaluated on the same held-out districts, so this ranking is trustworthy — no split leaked across models.

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('F1_macro', ascending=False)
results_df = results_df.round(4)
results_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
results_df[['Accuracy','F1_macro','ROC_AUC_ovr']].plot(kind='bar', ax=ax, colormap='viridis')
plt.title('Hybrid Model Comparison — Accuracy vs F1-macro vs ROC-AUC')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

best_model_name = results_df['F1_macro'].idxmax()
print(f'Best performing hybrid by F1-macro: {best_model_name}')


## 7. Explainability with SHAP

Uses the XGBoost model trained inside Hybrid A's stacking step (`base_models['xgb']`).

In [ ]:
xgb_for_shap = base_models['xgb']
explainer = shap.TreeExplainer(xgb_for_shap)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    sv_for_plot = shap_values[int(np.argmax(np.bincount(y_test)))]
elif shap_values.ndim == 3:
    sv_for_plot = shap_values[:, :, int(np.argmax(np.bincount(y_test)))]
else:
    sv_for_plot = shap_values

shap.summary_plot(sv_for_plot, X_test, show=False)
plt.title('SHAP Summary — XGBoost (dominant class)')
plt.tight_layout()
plt.show()


In [ ]:
shap.summary_plot(sv_for_plot, X_test, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance — XGBoost')
plt.tight_layout()
plt.show()
